# Clasificación de Residuos con Transfer Learning (VGG16)

Clasificador de imágenes para distinguir entre residuos **orgánicos (O)** y **reciclables (R)** usando Transfer Learning con VGG16 pre-entrenado en ImageNet.

Se entrenan dos versiones:
- **Extracción de características**: todas las capas de VGG16 congeladas.
- **Fine-tuning**: última capa convolucional descongelada para mejorar el rendimiento.

In [ ]:
# Instalación de las librerías necesarias con versiones específicas
!pip install tensorflow==2.17.0 -q
!pip install numpy -q
!pip install scikit-learn==1.5.1 -q
!pip install matplotlib==3.9.2 -q
!pip install tqdm -q

In [ ]:
# Importación de librerías
import numpy as np
import os
import glob

from matplotlib import pyplot as plt

# Suprimir mensajes de log de TensorFlow (solo errores críticos)
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'

import tensorflow as tf
from tensorflow.keras.models import Sequential, Model
from tensorflow.keras import optimizers
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint
from tensorflow.keras.layers import Dense, Dropout, Flatten
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from sklearn import metrics

import warnings
warnings.filterwarnings('ignore')

In [ ]:
# Task 1: Verificar la versión de TensorFlow instalada
print('Versión de TensorFlow:', tf.__version__)

In [ ]:
# Descarga y extracción del dataset (residuos orgánicos vs reciclables)
import requests
import zipfile
from tqdm import tqdm

url = 'https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/kd6057VPpABQ2FqCbgu9YQ/o-vs-r-split-reduced-1200.zip'
file_name = 'o-vs-r-split-reduced-1200.zip'

print('Descargando dataset...')
with requests.get(url, stream=True) as response:
    response.raise_for_status()
    with open(file_name, 'wb') as f:
        for chunk in response.iter_content(chunk_size=8192):
            f.write(chunk)

# Extraer el zip mostrando barra de progreso
print('Extrayendo archivos...')
with zipfile.ZipFile(file_name, 'r') as zip_ref:
    members = zip_ref.infolist()
    with tqdm(total=len(members), unit='file') as progress_bar:
        for member in members:
            zip_ref.extract(member)
            progress_bar.update(1)

# Eliminar el zip para liberar espacio
os.remove(file_name)
print('Dataset listo.')

In [ ]:
# Parámetros de configuración del modelo y datos
img_rows, img_cols = 150, 150   # Dimensiones de entrada de las imágenes
batch_size = 32                  # Imágenes por lote durante el entrenamiento
n_epochs = 10                    # Número máximo de épocas
n_classes = 2                    # Clases: Orgánico (O) y Reciclable (R)
val_split = 0.2                  # 20% de los datos de entrenamiento para validación
path = 'o-vs-r-split/train/'    # Ruta al conjunto de entrenamiento
path_test = 'o-vs-r-split/test/' # Ruta al conjunto de prueba
input_shape = (img_rows, img_cols, 3)  # Shape: (alto, ancho, canales RGB)
labels = ['O', 'R']             # Etiquetas de clases
seed = 42                        # Semilla para reproducibilidad

In [ ]:
# Creación de generadores de imágenes
# El ImageDataGenerator aplica augmentation (transformaciones aleatorias) al conjunto
# de entrenamiento para que el modelo generalice mejor con menos imágenes

# Entrenamiento: augmentation + normalización
train_datagen = ImageDataGenerator(
    validation_split=val_split,   # Reservar 20% para validación
    rescale=1.0/255.0,            # Normalizar píxeles al rango [0, 1]
    width_shift_range=0.1,        # Desplazamiento horizontal aleatorio (±10%)
    height_shift_range=0.1,       # Desplazamiento vertical aleatorio (±10%)
    horizontal_flip=True          # Volteo horizontal aleatorio
)

# Validación: solo normalización, sin augmentation
val_datagen = ImageDataGenerator(
    validation_split=val_split,
    rescale=1.0/255.0
)

# Prueba: solo normalización
test_datagen = ImageDataGenerator(
    rescale=1.0/255.0
)

In [ ]:
# Generador de datos de entrenamiento (80% de las imágenes en /train)
train_generator = train_datagen.flow_from_directory(
    directory=path,
    seed=seed,
    batch_size=batch_size,
    class_mode='binary',            # Clasificación binaria: O vs R
    shuffle=True,                    # Mezclar imágenes en cada época
    target_size=(img_rows, img_cols),
    subset='training'               # Subconjunto de entrenamiento
)

In [ ]:
# Generador de datos de validación (20% de las imágenes en /train)
val_generator = val_datagen.flow_from_directory(
    directory=path,
    seed=seed,
    batch_size=batch_size,
    class_mode='binary',
    shuffle=True,
    target_size=(img_rows, img_cols),
    subset='validation'             # Subconjunto de validación
)

In [ ]:
# Task 2: Generador de datos de prueba
# shuffle=False para mantener el orden y comparar predicciones con etiquetas reales
test_generator = test_datagen.flow_from_directory(
    directory=path_test,
    seed=seed,
    batch_size=batch_size,
    class_mode='binary',
    shuffle=False,                  # Sin mezclar para conservar el orden de etiquetas
    target_size=(img_rows, img_cols)
)

In [ ]:
# Task 3: Longitud del generador de entrenamiento
# len() devuelve la cantidad de lotes (batches) disponibles por época
print('Número de lotes en train_generator:', len(train_generator))

In [ ]:
# Visualización de data augmentation: 5 versiones aumentadas de una misma imagen
from pathlib import Path

IMG_DIM = (100, 100)

# Cargar las primeras 20 imágenes de la clase O (orgánico)
train_files = glob.glob('./o-vs-r-split/train/O/*')[:20]
train_imgs = [
    tf.keras.preprocessing.image.img_to_array(
        tf.keras.preprocessing.image.load_img(img, target_size=IMG_DIM)
    )
    for img in train_files
]
train_imgs = np.array(train_imgs)
train_labels = [Path(fn).parent.name for fn in train_files]

# Generar 5 versiones aumentadas de la primera imagen
img_id = 0
O_generator = train_datagen.flow(
    train_imgs[img_id:img_id+1],
    train_labels[img_id:img_id+1],
    batch_size=1
)
O = [next(O_generator) for i in range(5)]

# Mostrar las 5 versiones
fig, ax = plt.subplots(1, 5, figsize=(16, 6))
fig.suptitle('Ejemplos de Data Augmentation aplicado a la clase O', fontsize=13)
print('Etiquetas:', [item[1][0] for item in O])
for i in range(5):
    ax[i].imshow(O[i][0][0])
    ax[i].axis('off')
plt.tight_layout()
plt.show()

In [ ]:
# Carga del modelo VGG16 pre-entrenado en ImageNet
# include_top=False: se eliminan las capas densas originales,
# conservando solo el extractor de características convolucional
from tensorflow.keras.applications import vgg16

input_shape = (150, 150, 3)
vgg = vgg16.VGG16(
    include_top=False,         # Sin el clasificador original de 1000 clases
    weights='imagenet',        # Pesos pre-entrenados en ImageNet
    input_shape=input_shape
)

# Aplanar la salida del último bloque convolucional para conectar capas densas
output = vgg.layers[-1].output
output = tf.keras.layers.Flatten()(output)

# Crear el modelo base (entradas y salidas de VGG16)
basemodel = Model(vgg.input, output)

In [ ]:
# Congelar todas las capas del modelo base
# Los pesos de VGG16 NO se actualizarán durante el entrenamiento;
# solo aprenderán las capas densas que agregamos encima
for layer in basemodel.layers:
    layer.trainable = False

print(f'Capas del modelo base congeladas: {len(basemodel.layers)}')

In [ ]:
# Construcción del modelo completo sobre VGG16
# Se agregan capas densas para adaptar la clasificación a nuestro dataset binario
model = Sequential()
model.add(basemodel)                        # Extractor de características VGG16
model.add(Dense(512, activation='relu'))    # Capa densa con activación ReLU
model.add(Dropout(0.3))                     # Dropout 30% para reducir overfitting
model.add(Dense(512, activation='relu'))    # Segunda capa densa
model.add(Dropout(0.3))
model.add(Dense(1, activation='sigmoid'))   # Salida: probabilidad de clase R (0-1)

In [ ]:
# Task 4: Resumen de la arquitectura del modelo
# Muestra capas, shape de salida y número de parámetros entrenables vs no entrenables
model.summary()

In [ ]:
# Asegurar que las capas del modelo base siguen congeladas antes de compilar
for layer in basemodel.layers:
    layer.trainable = False

# Task 5: Compilar el modelo
model.compile(
    loss='binary_crossentropy',                         # Pérdida para clasificación binaria
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-5),  # Adam con learning rate bajo
    metrics=['accuracy']                                # Métrica a monitorear
)

In [ ]:
# Callbacks para el entrenamiento:
# - LossHistory_: registra pérdida y learning rate por época
# - LearningRateScheduler: aplica decaimiento exponencial al learning rate
# - EarlyStopping: detiene si val_loss no mejora en 4 épocas consecutivas
# - ModelCheckpoint: guarda solo el mejor modelo según val_loss
from tensorflow.keras.callbacks import LearningRateScheduler

checkpoint_path = 'O_R_tlearn_vgg16.keras'

# Callback personalizado: guarda histórico de pérdida y learning rate
class LossHistory_(tf.keras.callbacks.Callback):
    def on_train_begin(self, logs={}):
        self.losses = []
        self.lr = []

    def on_epoch_end(self, epoch, logs={}):
        self.losses.append(logs.get('loss'))
        self.lr.append(exp_decay(epoch))
        print('lr:', exp_decay(len(self.losses)))

# Decaimiento exponencial: lr = lr_0 * e^(-k * epoch)
def exp_decay(epoch):
    initial_lrate = 1e-4
    k = 0.1
    return initial_lrate * np.exp(-k * epoch)

# Instanciar todos los callbacks
loss_history_ = LossHistory_()
lrate_ = LearningRateScheduler(exp_decay)

keras_callbacks = [
    EarlyStopping(monitor='val_loss', patience=4, mode='min', min_delta=0.01),
    ModelCheckpoint(checkpoint_path, monitor='val_loss', save_best_only=True, mode='min')
]

callbacks_list_ = [loss_history_, lrate_] + keras_callbacks

# Entrenamiento: extracción de características (capas VGG16 congeladas)
print('Entrenando modelo de extracción de características...')
extract_feat_model = model.fit(
    train_generator,
    steps_per_epoch=len(train_generator),  # Todos los lotes disponibles por época
    epochs=10,
    callbacks=callbacks_list_,
    validation_data=val_generator,
    validation_steps=val_generator.samples // batch_size,
    verbose=1
)

In [ ]:
# Curvas de pérdida del modelo de extracción de características
# Si val_loss > loss, hay overfitting; si ambas bajan, el modelo está aprendiendo bien
history = extract_feat_model

plt.figure(figsize=(5, 5))
plt.plot(history.history['loss'], label='Pérdida Entrenamiento')
plt.plot(history.history['val_loss'], label='Pérdida Validación')
plt.title('Curva de Pérdida - Extracción de Características')
plt.xlabel('Épocas')
plt.ylabel('Pérdida')
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
# Task 6: Curvas de accuracy del modelo de extracción de características
history = extract_feat_model

plt.figure(figsize=(5, 5))
plt.plot(history.history['accuracy'], label='Accuracy Entrenamiento')
plt.plot(history.history['val_accuracy'], label='Accuracy Validación')
plt.title('Curva de Accuracy - Extracción de Características')
plt.xlabel('Épocas')
plt.ylabel('Accuracy')
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
# ================================================================
# FINE-TUNING: Descongelar la última capa convolucional del VGG16
# Permitir que block5_conv3 se ajuste al dataset de residuos
# mejora el rendimiento respecto a la extracción pura de características
# ================================================================
from tensorflow.keras.applications import vgg16

input_shape = (150, 150, 3)
vgg = vgg16.VGG16(include_top=False, weights='imagenet', input_shape=input_shape)

output = vgg.layers[-1].output
output = tf.keras.layers.Flatten()(output)
basemodel = Model(vgg.input, output)

# Primero congelar todo el modelo base
for layer in basemodel.layers:
    layer.trainable = False

# Luego descongelar desde block5_conv3 en adelante
set_trainable = False
for layer in basemodel.layers:
    if layer.name == 'block5_conv3':
        set_trainable = True       # Activar descongelamiento a partir de esta capa
    if set_trainable:
        layer.trainable = True    # Esta capa y las siguientes serán entrenables

# Mostrar qué capas quedan entrenables
print('Estado de capas entrenables:')
for layer in basemodel.layers:
    print(f'  {layer.name}: {layer.trainable}')

In [ ]:
# Construcción y entrenamiento del modelo con fine-tuning
model = Sequential()
model.add(basemodel)
model.add(Dense(512, activation='relu'))
model.add(Dropout(0.3))
model.add(Dense(512, activation='relu'))
model.add(Dropout(0.3))
model.add(Dense(1, activation='sigmoid'))

checkpoint_path = 'O_R_tlearn_fine_tune_vgg16.keras'

# Reutilizar callbacks definidos anteriormente
loss_history_ = LossHistory_()
lrate_ = LearningRateScheduler(exp_decay)

keras_callbacks = [
    EarlyStopping(monitor='val_loss', patience=4, mode='min', min_delta=0.01),
    ModelCheckpoint(checkpoint_path, monitor='val_loss', save_best_only=True, mode='min')
]

callbacks_list_ = [loss_history_, lrate_] + keras_callbacks

# Compilar con RMSprop y learning rate más bajo para no destruir los pesos pre-entrenados
model.compile(
    loss='binary_crossentropy',
    optimizer=optimizers.RMSprop(learning_rate=1e-4),
    metrics=['accuracy']
)

print('Entrenando modelo con fine-tuning...')
fine_tune_model = model.fit(
    train_generator,
    steps_per_epoch=len(train_generator),  # Todos los lotes disponibles por época
    epochs=10,
    callbacks=callbacks_list_,
    validation_data=val_generator,
    validation_steps=val_generator.samples // batch_size,
    verbose=1
)

In [ ]:
# Task 7: Curvas de pérdida del modelo fine-tuning
history = fine_tune_model

plt.figure(figsize=(5, 5))
plt.plot(history.history['loss'], label='Pérdida Entrenamiento')
plt.plot(history.history['val_loss'], label='Pérdida Validación')
plt.title('Curva de Pérdida - Fine-Tuning')
plt.xlabel('Épocas')
plt.ylabel('Pérdida')
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
# Task 8: Curvas de accuracy del modelo fine-tuning
history = fine_tune_model

plt.figure(figsize=(5, 5))
plt.plot(history.history['accuracy'], label='Accuracy Entrenamiento')
plt.plot(history.history['val_accuracy'], label='Accuracy Validación')
plt.title('Curva de Accuracy - Fine-Tuning')
plt.xlabel('Épocas')
plt.ylabel('Accuracy')
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
# Evaluación de ambos modelos en el conjunto de prueba
from pathlib import Path

# Cargar los mejores modelos guardados durante el entrenamiento
extract_feat_model = tf.keras.models.load_model('O_R_tlearn_vgg16.keras')
fine_tune_model    = tf.keras.models.load_model('O_R_tlearn_fine_tune_vgg16.keras')

IMG_DIM = (150, 150)

# Cargar 50 imágenes de prueba de cada clase
test_files_O = glob.glob('./o-vs-r-split/test/O/*')
test_files_R = glob.glob('./o-vs-r-split/test/R/*')
test_files   = test_files_O[:50] + test_files_R[:50]

test_imgs = [
    tf.keras.preprocessing.image.img_to_array(
        tf.keras.preprocessing.image.load_img(img, target_size=IMG_DIM)
    )
    for img in test_files
]
test_imgs   = np.array(test_imgs)
test_labels = [Path(fn).parent.name for fn in test_files]

# Normalizar píxeles al rango [0, 1]
test_imgs_scaled = test_imgs.astype('float32') / 255

# Convertir etiquetas texto <-> número (O=0, R=1 basado en salida sigmoide)
num2class_lt = lambda l: ['O' if x < 0.5 else 'R' for x in l]

# Predicciones de ambos modelos
predictions_extract  = num2class_lt(extract_feat_model.predict(test_imgs_scaled, verbose=0))
predictions_finetune = num2class_lt(fine_tune_model.predict(test_imgs_scaled, verbose=0))

# Reporte de clasificación: precisión, recall y F1-score por clase
print('=== Modelo Extracción de Características ===')
print(metrics.classification_report(test_labels, predictions_extract))

print('=== Modelo Fine-Tuning ===')
print(metrics.classification_report(test_labels, predictions_finetune))

In [ ]:
# Función auxiliar para visualizar una imagen con su etiqueta real y predicha
def plot_image_with_title(image, model_name, actual_label, predicted_label):
    plt.imshow(image)
    correct = actual_label == predicted_label
    color = 'green' if correct else 'red'
    plt.title(
        f'Modelo: {model_name}\nReal: {actual_label} | Predicho: {predicted_label}',
        color=color
    )
    plt.axis('off')
    plt.show()

In [ ]:
# Task 9: Visualizar imagen de prueba con predicción del modelo de extracción (índice 1)
index_to_plot = 1

plot_image_with_title(
    image=test_imgs[index_to_plot].astype('uint8'),
    model_name='Extracción de Características',
    actual_label=test_labels[index_to_plot],
    predicted_label=predictions_extract[index_to_plot]
)

In [ ]:
# Task 10: Visualizar imagen de prueba con predicción del modelo fine-tuning (índice 1)
index_to_plot = 1

plot_image_with_title(
    image=test_imgs[index_to_plot].astype('uint8'),
    model_name='Fine-Tuning',
    actual_label=test_labels[index_to_plot],
    predicted_label=predictions_finetune[index_to_plot]
)